In [7]:
import numpy as np
import pandas as pd
pd.set_option('display.float_format', lambda x: '%.13f' % x)
pd.set_option('display.max_columns', None)

from scipy.stats import wilcoxon

In [8]:
df = pd.read_excel('all-results.xlsx', index_col=0)
df.head()

,DTW_D,ROCKET,HC2,Hydra,MR+Hydra,DLinear,LightTS,MTS-Mixer,TimesNet,ModernTCN,TSLANet,TimeMixer++,FEDformer,ETSformer,Crossformer,PatchTST,GPT4TS,iTransformer,InterpGN,TSCMamba,MambaSL
EC,32.3193916349810,44.4866920152091,52.8517110266160,54.3726235741445,60.0760456273764,31.1787072243346,32.3193916349810,30.7984790874525,33.4600760456274,32.3193916349810,31.9391634980989,34.6007604562738,33.8403041825095,33.0798479087452,43.3460076045627,32.6996197718631,32.3193916349810,31.9391634980989,28.8973384030418,30.7984790874525,42.5855513307985
FD,52.8660612939841,61.6061293984109,61.3791146424518,60.9818388195233,61.2939841089671,69.4948921679909,68.4733257661748,68.5300794551646,70.1475595913734,66.9977298524404,66.9693530079455,69.6651532349603,69.8921679909194,68.3314415437004,69.6367763904654,68.0760499432463,68.6152099886493,68.4165720771850,63.7911464245176,70.2043132803632,69.2962542565267
HW,60.7058823529412,59.2941176470588,56.0000000000000,56.3529411764706,53.5294117647059,23.7647058823529,25.0588235294118,19.6470588235294,37.5294117647059,30.1176470588235,62.0000000000000,33.6470588235294,33.4117647058824,35.6470588235294,30.3529411764706,29.5294117647059,34.1176470588235,31.2941176470588,58.7058823529412,45.0588235294118,60.8235294117647
HB,71.7073170731707,75.6097560975610,78.5365853658537,76.5853658536585,77.0731707317073,76.5853658536585,77.5609756097561,79.0243902439024,83.9024390243902,78.0487804878049,79.5121951219512,80.4878048780488,78.0487804878049,78.5365853658537,79.0243902439024,73.6585365853659,79.0243902439024,78.0487804878049,80.4878048780488,78.5365853658537,80.0000000000000
JV,95.9459459459459,83.7837837837838,98.3783783783784,97.8378378378378,98.3783783783784,96.4864864864865,96.7567567567568,95.9459459459459,98.6486486486486,98.6486486486486,98.9189189189189,99.1891891891892,97.5675675675676,98.3783783783784,98.3783783783784,97.0270270270270,98.1081081081081,98.1081081081081,99.7297297297297,98.3783783783784,98.6486486486486


In [9]:
df_norm = ((df.T - df.mean(axis=1)) / df.std(axis=1)).T
wilcoxon_res = list()
for c in df.columns:
    if c == 'MambaSL':
        continue
    res = wilcoxon((df_norm['MambaSL'] - df_norm[c]), alternative='greater')
    wilcoxon_res.append((c, res.pvalue))

df_wilcoxon = pd.DataFrame(wilcoxon_res, columns=['Classifier', 'p-value']).set_index('Classifier')
df_wilcoxon.T

Classifier,DTW_D,ROCKET,HC2,Hydra,MR+Hydra,DLinear,LightTS,MTS-Mixer,TimesNet,ModernTCN,TSLANet,TimeMixer++,FEDformer,ETSformer,Crossformer,PatchTST,GPT4TS,iTransformer,InterpGN,TSCMamba
p-value,0.0000036148028,0.0008239997984,0.0562140298397,0.0339326687995,0.0152596244667,0.0000017574815,0.0001029188644,0.0000049888494,0.0041271717168,0.0011389955577,0.0397302551879,0.0065308029432,0.0000849487260,0.0001042951466,0.0058827127747,0.0004876820683,0.0001884871277,0.0001750380135,0.0327848970781,0.0413265116341


In [10]:
df_rank = df.rank(axis=1, method='min',ascending=False).astype(int)
df_rank.head()

,DTW_D,ROCKET,HC2,Hydra,MR+Hydra,DLinear,LightTS,MTS-Mixer,TimesNet,ModernTCN,TSLANet,TimeMixer++,FEDformer,ETSformer,Crossformer,PatchTST,GPT4TS,iTransformer,InterpGN,TSCMamba,MambaSL
EC,12,4,3,2,1,18,12,19,9,12,16,7,8,10,5,11,12,16,21,19,6
FD,21,17,18,20,19,6,10,9,2,14,15,4,3,12,5,13,8,11,16,1,7
HW,3,4,7,6,8,20,19,21,10,17,1,13,14,11,16,18,12,15,5,9,2
HB,21,19,9,17,16,17,15,6,1,12,5,2,12,9,6,20,6,12,2,9,4
JV,19,21,7,14,7,18,17,19,4,4,3,2,15,7,7,16,12,12,1,7,4


In [11]:
labels = df.columns
scores = df.to_numpy()
ranks = df_rank.to_numpy()

n_datasets, n_estimators = df.shape

avg_ranks = ranks.mean(axis=0)
ordered_labels_ranks = np.array(
    [(l, float(r)) for r, l in sorted(zip(avg_ranks, labels))], dtype=object
)
ordered_labels = np.array([la for la, _ in ordered_labels_ranks], dtype=str)
ordered_avg_ranks = np.array([r for _, r in ordered_labels_ranks], dtype=np.float32)
ordered_ranks = ranks[:, [np.where(np.array(labels) == r)[0][0] for r in ordered_labels]]

print(ordered_labels_ranks)

[['MambaSL' 5.366666666666666]
 ['TimeMixer++' 7.033333333333333]
 ['TSCMamba' 7.1]
 ['TSLANet' 7.8]
 ['InterpGN' 7.866666666666666]
 ['Crossformer' 8.5]
 ['TimesNet' 8.666666666666666]
 ['Hydra' 8.833333333333334]
 ['HC2' 8.9]
 ['ModernTCN' 9.133333333333333]
 ['MR+Hydra' 9.4]
 ['GPT4TS' 9.5]
 ['ETSformer' 10.633333333333333]
 ['ROCKET' 11.2]
 ['LightTS' 11.833333333333334]
 ['iTransformer' 11.833333333333334]
 ['FEDformer' 12.333333333333334]
 ['PatchTST' 12.6]
 ['MTS-Mixer' 13.9]
 ['DTW_D' 16.033333333333335]
 ['DLinear' 16.8]]
